In [ ]:
import os
import click

from lotka_volterra_UDE_case_study.mod import Func
import jax.random as jrandom
import jax.numpy as jnp
import xarray as xr
from pymob.simulation import SimulationBase
from pymob.solvers.diffrax import UDESolver
from pymob.sim.config import Param
import diffrax
import jax.random as jr
import jax

jax.config.update('jax_enable_x64', True)

c:\Users\Markus\anaconda3\envs\pymob3\Lib\site-packages\sympy2jax\sympy_module.py:290: UserWarning: `equinox.static_field` is deprecated in favour of `equinox.field(static=True)`
  has_extra_funcs: bool = eqx.static_field()


In [ ]:
def _get_data(ts, theta, max, min, noisiness, *, key):
    """
    Returns a single time series (evaluated at the time points defined by ts) of the 
    Lotka-Volterra model with some normally-distributed noise. Initial conditions for 
    prey and predator are both chosen randomly from the range [min, max].

    Parameters
    ----------
    ts : jax.ArrayImpl
        An array containing all the time points the timeseries should be evaluated for.
    theta : list
        A list of four floats representing the parameters of the Lotka Volterra model
        [alpha, beta, gamma, delta].
    max : float
        Maximum value for the initial prey and predator values (before adding noise).
    min : float
        Minimum value for the initial prey and predator values (before adding noise).
    noisiness : float
        Scale of the normal distribution the noise is drawn from. I fnoisiness == 0,
        no noise is added.
    key : jax.ArrayImpl, optional
        A key used to make stochastic processes (in this case the noise values drawn 
        from a normal distribution) reproducible. If no key is provided, noise may
        differ between runs.

    Returns:
    --------
    jax.ArrayImpl
        An array containing a noisy Lotka Volterra time series, evaluated at time
        points ts.
    """
    
    y0 = jr.uniform(key, (2,), minval=min, maxval=max)

    def f(t, y, args):
        dXdt = theta[0] * y[0] - theta[1] * y[0] * y[1]
        dYdt = theta[2] * y[0] * y[1] - theta[3] * y[1]
        return jnp.stack([dXdt, dYdt], axis=-1)

    solver = diffrax.Tsit5()
    dt0 = 0.1
    saveat = diffrax.SaveAt(ts=ts)
    sol = diffrax.diffeqsolve(
        diffrax.ODETerm(f), solver, ts[0], ts[-1], dt0, y0, saveat=saveat
    )
    ys = sol.ys
    noise = jr.normal(key=key, shape=(len(ts), 2))
    ys += noisiness * noise
    return jnp.greater(ys, jnp.zeros(ys.shape)) * ys + 1e-8

def get_data(dataset_size, theta, max, min, t_end, datapoints, noisiness, *, key):
    """
    Returns multiple time series (evaluated at the time points defined by ts) of the 
    Lotka-Volterra model with some normally-distributed noise and different initial 
    conditions for prey and predator chosen randomly from the range [min, max].

    Parameters
    ----------
    dataset_size : int
        The amount of generated time series.
    theta : list
        A list of four floats representing the parameters of the Lotka Volterra model
        [alpha, beta, gamma, delta].
    max : float
        Maximum value for the initial prey and predator values (before adding noise).
    min : float
        Minimum value for the initial prey and predator values (before adding noise).
    t_end : float
        The last point in time that the time series are supposed to contain.
    datapoints : int
        The amount of evenly-spaced datapoints each time series is supposed to contain.
    noisiness : float
        Scale of the normal distribution the noise is drawn from. If noisiness == 0,
        no noise is added.
    key : jax.Array
        A key used to make stochastic processes (in this case the noise values drawn 
        from a normal distribution) reproducible. If no key is provided, noise may
        differ between runs.

    Returns:
    --------
    jax.ArrayImpl
        An array containing multiple noisy Lotka Volterra time series, evaluated at time
        points ts.
    """

    ts = jnp.linspace(0, t_end, datapoints)
    key = jr.split(key, dataset_size)
    ys = jax.vmap(lambda key: _get_data(ts, theta, max, min, noisiness, key=key))(key)
    return ts, ys

In [ ]:
length_strategy = [0.1, 1]
lr_strategy = [1e-3, 1e-3]
optim_strategy = ["adabelief(learning_rate=1e-3)"] * 2
clip_strategy = [0.1, 0]
batch_size = 10
data_points = 100
data_noise = 0.2

# Create simulation base and model
sim = SimulationBase()
sim.config.case_study.name = "lotka_volterra_UDE_case_study"
sim.config.case_study.scenario = "UDETest"
key = jrandom.PRNGKey(5678)
data_key, model_key, loader_key = jrandom.split(key, 3)
sim.model = Func({"alpha":jnp.array([1.3]), "delta":jnp.array([1.8])},key=model_key)

# Add synthetic observation data and initial conditions
ts,ys = get_data(50, [1.3,0.9,0.8,1.8], 5, 1, 50, data_points, data_noise, key=jr.PRNGKey(0))
datasets = jnp.linspace(0, 49, 50)
test_data1 = xr.DataArray(ys[:,:,0], coords={"batch_id": datasets, "time": ts}).to_dataset(name="prey")
test_data2 = xr.DataArray(ys[:,:,1], coords={"batch_id": datasets, "time": ts}).to_dataset(name="predator")
test_data = xr.merge([test_data1, test_data2])
sim.observations = test_data
sim.model_parameters["y0"] = sim.observations.sel(time = 0).drop_vars("time")

# Define model parameters
sim.config.model_parameters.alpha = Param(value=1.3, free=False)
sim.config.model_parameters.delta = Param(value=1.8, free=True)
sim.config.model_parameters.delta.prior = "uniform(loc=1.0,scale=2.0)"

# Set solver settings and dispatch evaluator
sim.solver = UDESolver
sim.config.jaxsolver.throw_exception = False
sim.dispatch_constructor()
evaluator = sim.dispatch()

# Determine inferer settings, create inferer, and run it
sim.config.inference_optax.MLP_weight_dist = "normal()"
sim.config.inference_optax.MLP_bias_dist = "normal()"
sim.config.inference_optax.batch_size = batch_size
sim.config.inference_optax.data_split = 0.8
sim.config.inference_optax.multiple_runs_target = 1
sim.config.inference_optax.multiple_runs_limit = 100
sim.config.inference_optax.time_limit = 1200
sim.config.inference_optax.indepth = "partial"
sim.config.inference_optax.length_strategy = [i for i in length_strategy if i != -1]
sim.config.inference_optax.steps_strategy = [4000/len(sim.config.inference_optax.length_strategy) for i in sim.config.inference_optax.length_strategy]
sim.config.inference_optax.lr_strategy = lr_strategy
sim.config.inference_optax.optim_strategy = optim_strategy
sim.config.inference_optax.clip_strategy = clip_strategy
sim.set_inferer("optax")
sim.inferer.run()

C:\Users\Markus\pymob\pymob\simulation.py:309: UserWarning: `sim.config.data_structure.prey = Datavariable(dimensions=['batch_id', 'time'] min=1e-08 max=6.979921372971042 observed=True dimensions_evaluator=None)` has been assumed from `sim.observations`. If the order of the dimensions should be different, specify `sim.config.data_structure.prey = DataVariable(dimensions=[...], ...)` manually.
  warnings.warn(
C:\Users\Markus\pymob\pymob\simulation.py:309: UserWarning: `sim.config.data_structure.predator = Datavariable(dimensions=['batch_id', 'time'] min=1e-08 max=5.189636850648957 observed=True dimensions_evaluator=None)` has been assumed from `sim.observations`. If the order of the dimensions should be different, specify `sim.config.data_structure.predator = DataVariable(dimensions=[...], ...)` manually.
  warnings.warn(
C:\Users\Markus\pymob\pymob\simulation.py:579: UserWarning: The number of ODE states was not specified in the config file [simulation] > 'n_ode_states = <n>'. Extract

MinMaxScaler(variable=prey, min=1e-08, max=6.979921372971042)
MinMaxScaler(variable=predator, min=1e-08, max=5.189636850648957)


0 of 1 runs completed:  15%|█▍        | 598/4000 [04:30<01:19, 42.99it/s, 5 unsuccessful runs so far]   

#### Debugging L-BFGS

In [4]:
import optax
import equinox as eqx
import functools
import jax.tree_util as jtu

In [5]:
@eqx.filter_value_and_grad
def grad_loss(model, ti, yi, mask, t_thresh, x_in, loss_func):
    """
    Computes the loss and gradients of all model parameters.

    Parameters
    ----------
    model : Any
        Model object containing all model parameters somewhere within a pytree
        structure and returning derivatives depending on the model states when
        called with the model states as input.
    ti : jax.ArrayImpl
        A jax.numpy array containing the coordinates along the time axis.
    yi : jax.ArrayImpl
        A jax.numpy array containing a batch of observation data.
    mask : jax.ArrayImpl
        An array containing True for all observations that should be considered
        during loss calculation, and False for all other observations.
    t_thresh : float
        The point in time up to which model predictions are calculated.
    x_in : jax.ArrayImpl
        A jax.numpy array containing a batch of input data.
    """
    y_pred = jnp.array(jax.vmap(sim.evaluator._solver.standalone_solver, in_axes=(None, None, 0, None, None))(model, ti, yi[:, 0], x_in, t_thresh))
    y_pred = jnp.stack(y_pred, axis = (len(y_pred.shape)-1))

    losses = loss_func(yi, y_pred)

    return jnp.mean(jnp.where(mask, losses, mask))

In [6]:
def loss_func(y_obs, y_pred):
    return sim.model.loss(jnp.where(jnp.isnan(y_obs), y_pred, y_obs), y_pred)
length_eval = 100
mask = jnp.concatenate([jnp.ones(ys[:10].shape)[:batch_size,:length_eval], jnp.zeros(ys[:10].shape)[:batch_size, length_eval :]], axis=1)
loss, grad = grad_loss(Func({"alpha":jnp.array(1.3), "delta":jnp.array(1.8)},key=model_key), ts, ys[:10], mask, 50, None, loss_func)

In [7]:
optim = optax.lbfgs()
opt_state = optim.init(eqx.filter(sim.model, eqx.is_inexact_array))

In [8]:
updates, opt_state = optim.update(grad, opt_state, eqx.filter(sim.model, eqx.is_inexact_array), value=loss, grad=grad, value_fn=grad_loss)

TypeError: scan body function carry input and carry output must have equal types, but they differ:

  * the input carry component vec.UDE_params[0][1] has type float64[] but the corresponding output carry component has type float64[1], so the shapes do not match;

  * the input carry component vec.UDE_params[1][1] has type float64[] but the corresponding output carry component has type float64[1], so the shapes do not match;

  * the input carry component vec.alpha has type float64[] but the corresponding output carry component has type float64[1], so the shapes do not match;
  * the input carry component vec.delta has type float64[] but the corresponding output carry component has type float64[1], so the shapes do not match.

Revise the function so that all output types match the corresponding input types.

#### Debugging SM3

In [4]:
import optax
import equinox as eqx
import functools
import jax.tree_util as jtu

In [5]:
optim = optax.sm3(**{key: entry.evaluate() for key, entry in sim.config.inference_optax.optim_strategy[0].parameters.items()})

In [6]:
opt_state = optim.init(eqx.filter(sim.model, eqx.is_inexact_array))

In [7]:
def zeros_for_dim(p):
    return [jnp.zeros([s], dtype=p.dtype) for s in p.shape]

jax.tree.map(zeros_for_dim, eqx.filter(sim.model, eqx.is_inexact_array))

Func(
  UDE_params=[(None, [f64[1]]), (None, [f64[1]])],
  mlp=MLP(
    layers=(
      Linear(
        weight=[f64[3], f64[2]],
        bias=[f64[3]],
        in_features=2,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[3], f64[3]],
        bias=[f64[3]],
        in_features=3,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[3], f64[3]],
        bias=[f64[3]],
        in_features=3,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[2], f64[3]],
        bias=[f64[2]],
        in_features=3,
        out_features=2,
        use_bias=True
      )
    ),
    activation=None,
    final_activation=None,
    use_bias=True,
    use_final_bias=True,
    in_size=2,
    out_size=2,
    width_size=3,
    depth=3
  ),
  alpha=[f64[1]],
  delta=[f64[1]]
)

In [8]:
opt_state[0].mu

Func(
  UDE_params=[(None, [f64[1]]), (None, [f64[1]])],
  mlp=MLP(
    layers=(
      Linear(
        weight=[f64[3], f64[2]],
        bias=[f64[3]],
        in_features=2,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[3], f64[3]],
        bias=[f64[3]],
        in_features=3,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[3], f64[3]],
        bias=[f64[3]],
        in_features=3,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[2], f64[3]],
        bias=[f64[2]],
        in_features=3,
        out_features=2,
        use_bias=True
      )
    ),
    activation=None,
    final_activation=None,
    use_bias=True,
    use_final_bias=True,
    in_size=2,
    out_size=2,
    width_size=3,
    depth=3
  ),
  alpha=[f64[1]],
  delta=[f64[1]]
)

In [9]:
@eqx.filter_value_and_grad
def grad_loss(model, ti, yi, mask, t_thresh, x_in, loss_func):
    """
    Computes the loss and gradients of all model parameters.

    Parameters
    ----------
    model : Any
        Model object containing all model parameters somewhere within a pytree
        structure and returning derivatives depending on the model states when
        called with the model states as input.
    ti : jax.ArrayImpl
        A jax.numpy array containing the coordinates along the time axis.
    yi : jax.ArrayImpl
        A jax.numpy array containing a batch of observation data.
    mask : jax.ArrayImpl
        An array containing True for all observations that should be considered
        during loss calculation, and False for all other observations.
    t_thresh : float
        The point in time up to which model predictions are calculated.
    x_in : jax.ArrayImpl
        A jax.numpy array containing a batch of input data.
    """
    y_pred = jnp.array(jax.vmap(sim.evaluator._solver.standalone_solver, in_axes=(None, None, 0, None, None))(model, ti, yi[:, 0], x_in, t_thresh))
    y_pred = jnp.stack(y_pred, axis = (len(y_pred.shape)-1))

    losses = loss_func(yi, y_pred)

    return jnp.mean(jnp.where(mask, losses, mask))

In [14]:
def loss_func(y_obs, y_pred):
    return sim.model.loss(jnp.where(jnp.isnan(y_obs), y_pred, y_obs), y_pred)
length_eval = 100
mask = jnp.concatenate([jnp.ones(ys[:10].shape)[:batch_size,:length_eval], jnp.zeros(ys[:10].shape)[:batch_size, length_eval :]], axis=1)
loss, grad = grad_loss(Func({"alpha":jnp.array(1.3), "delta":jnp.array(1.8)},key=model_key), ts, ys[:10], mask, 50, None, loss_func)

In [15]:
b1 = 0.9
b2 = 1.0

def _expanded_shape(shape, axis):
    # Replaces a `shape` of [M, N, K] with 1 in all dimensions except for i.
    # For eg: i = 1 returns [1, N, 1].
    rank = len(shape)
    return [1] * axis + [shape[axis]] + [1] * (rank - axis - 1)

def _new_accum(g, v):
    coeffs = ((1.0 - b2) if b2 != 1.0 else 1.0, b2)
    if g.ndim < 2:
        return coeffs[0] * g**2 + coeffs[1] * v[0]
    else:
        return coeffs[0] * g**2 + coeffs[1] * functools.reduce(jnp.minimum, v)

def f(g, v):
    return [
        jnp.reshape(v[i], _expanded_shape(g.shape, i)) for i in range(g.ndim)
    ]

In [16]:
mu = jax.tree.map(f, grad, opt_state[0].mu)
accum = jax.tree.map(_new_accum, grad, mu)

IndexError: list index out of range

In [17]:
_new_accum(grad.mlp.layers[0].weight[2], mu.mlp.layers[0].weight[2])

IndexError: list index out of range

In [29]:
leaves, treedef = jtu.tree_flatten(grad, None)
all_leaves = [leaves] + [treedef.flatten_up_to(r) for r in [opt_state[0].mu]]
treedef.unflatten(_new_accum(*xs) for xs in zip(*all_leaves))

TypeError: min got incompatible shapes for broadcasting: (3,), (2,).

In [28]:
opt_state[0].mu

Func(
  UDE_params=[(None, [f64[1]]), (None, [f64[1]])],
  mlp=MLP(
    layers=(
      Linear(
        weight=[f64[3], f64[2]],
        bias=[f64[3]],
        in_features=2,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[3], f64[3]],
        bias=[f64[3]],
        in_features=3,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[3], f64[3]],
        bias=[f64[3]],
        in_features=3,
        out_features=3,
        use_bias=True
      ),
      Linear(
        weight=[f64[2], f64[3]],
        bias=[f64[2]],
        in_features=3,
        out_features=2,
        use_bias=True
      )
    ),
    activation=None,
    final_activation=None,
    use_bias=True,
    use_final_bias=True,
    in_size=2,
    out_size=2,
    width_size=3,
    depth=3
  ),
  alpha=[f64[1]],
  delta=[f64[1]]
)

In [32]:
_new_accum(*[xs for xs in zip(*all_leaves)][2])

TypeError: min got incompatible shapes for broadcasting: (3,), (2,).

In [119]:
opt_state[0].mu.UDE_params[0] = (None, 0)
opt_state[0].mu.UDE_params[0] = (None, 0)
opt_state[0].mu.alpha = 0
opt_state[0].mu.delta = 0

FrozenInstanceError: cannot assign to field 'alpha'

In [118]:
opt_state[0].mu.UDE_params

[(None, []), (None, [])]

In [70]:
jtu.tree_flatten(mu)

([Array([[0.],
         [0.],
         [0.]], dtype=float64),
  Array([[0., 0.]], dtype=float64),
  Array([0., 0., 0.], dtype=float64),
  Array([[0.],
         [0.],
         [0.]], dtype=float64),
  Array([[0., 0., 0.]], dtype=float64),
  Array([0., 0., 0.], dtype=float64),
  Array([[0.],
         [0.],
         [0.]], dtype=float64),
  Array([[0., 0., 0.]], dtype=float64),
  Array([0., 0., 0.], dtype=float64),
  Array([[0.],
         [0.]], dtype=float64),
  Array([[0., 0., 0.]], dtype=float64),
  Array([0., 0.], dtype=float64)],
 PyTreeDef(CustomNode(Func[('UDE_params', 'mlp', 'alpha', 'delta'), ()], [[(None, []), (None, [])], CustomNode(MLP[('layers', 'activation', 'final_activation'), (('use_bias', True), ('use_final_bias', True), ('in_size', 2), ('out_size', 2), ('width_size', 3), ('depth', 3))], [(CustomNode(Linear[('weight', 'bias'), (('in_features', 2), ('out_features', 3), ('use_bias', True))], [[*, *], [*]]), CustomNode(Linear[('weight', 'bias'), (('in_features', 3), ('out_f